# Thin lenses, zone plates, rings, axicons (EN)
The program enables the determination of the field distribution behind thin rotationally symmetric optical elements, such as lenses, Fresnel zone plates, axicons, and similar components, illuminated by either a plane wave or a Gaussian beam. The calculations employ the Hankel transform. One can choose either a scalar propagation model or a vectorial model with linear, radial, or azimuthal polarization. The computations are based on spatial-spectrum propagation using a transfer function corresponding to the Rayleigh–Sommerfeld propagator. For the scalar case, the Fresnel propagator can also be selected.

For each element, the full width at half maximum (FWHM) of the focal spot, or of the maximum energy density in the vicinity of the optical axis, can be determined. The computed FWHM is reliable only for sufficiently high sampling density. The size of the simulation domain depends on the parameter Nr, which determines the size of the Hankel-transform kernel, as well as on the sampling density; situations in which the field reaches the outer boundary of the simulation region should be avoided, as they lead to numerical artifacts.

For a binary Fresnel zone plate, the presence of multiple focal points at distances  f/(2m+1) can be observed. For a ring-shaped aperture, the formation of an on-axis maximum within the geometrical shadow region (a type of Poisson spot, also known as the Arago spot) is visible. Clear differences in the behavior of the optical elements depending on the polarization can also be observed.

# Soczewki cienkie / płytki strefowa Fresnela, aksikony (PL)
Program pozwala wyznaczyć rozkład pola za cienkimi elementemi optycznymi o symetrii obrotowej, takimi jak soczewki, płytki strefowe Fresnela i aksikony itp. oświetlane falą płaską albo wiązką Gaussa. Obliczenia wykorzystują transformatę Hankela. Można wybrać model propagacji skalarny, albo wektorowy z polaryzacją liniową, radialną lub azymutalną. Obliczenia wykorzystują propagację widma przestrzennego z funkcją przenoszenia odpowiadającą propagatorowi Rayleigha-Sommerfelda. Dla przypadku skalarnego można też wybrać propagator Fresnela.

Dla każdego z elementów można wyznaczyć rozmiar połówkowy (FWHM) ogniska, lub maksimum gęstości energii w pobliżu osi optycznej. Obliczony FWHM jest wiarygodny jedynie dla odpowiednio wysokiej gęstości próbkowania. Rozmiar obszaru symulacji zależy od liczby Nr decydującej o wielkosci jadra transformaty Hankela oraz od gęstości próbkowania i należy unikać sytuacji, gdy pole dochodzi do obszru zewnętrznego skutkujac powstawaniem artefaktów.

Dla binarnej płytki strefowej można zauważyć występowanie wielu ognisk w odległościach f/(2m+1). Dla apertury w kształcie pierścienia widać formowanie się na osi optycznej w obszarze cienia geometrycznego maksimum (rodzaj plamki Poissona, in. Arago). Widać też wyraźne róznice działania elementów w zależnosci od polaryzacji.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import special as sp
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [2]:
from IPython.display import display, HTML
import matplotlib as mpl

display(HTML("""
<style>
/* szerokość całego “pola” z widgetami */
.widget-box { width: 900px !important; }

/* szerokość dropdownów / sliderów / pól tekstowych */
.widget-dropdown, .widget-select, .widget-text, .widget-floattext, .widget-inttext,
.widget-slider, .widget-floatslider, .widget-intslider {
    width: 900px !important;
}

/* żeby opisy (description) się nie obcinały */
.widget-label, .widget-inline-hbox .widget-label {
    min-width: 260px !important;
}
</style>
"""))


In [3]:
# ----------------------------
# Helpers: math / optics
# ----------------------------
def build_mirrored_x_from_r(r):
    return np.concatenate((-r[:0:-1], r))

def mirror_profile_to_x(profile_r):
    return np.concatenate((profile_r[:0:-1], profile_r))

def kz_rs(k0, k_rho):
    # Correct branch: Im(kz) >= 0 ensures decay for evanescent components
    return np.sqrt((k0*k0 - k_rho*k_rho) + 0j)

def H_fresnel(k0, z, k_rho):
    return np.exp(-1j * z * (k_rho*k_rho) / (2.0*k0))

def H_rs(k0, z, k_rho):
    kz = kz_rs(k0, k_rho)
    return np.exp(1j * z * kz), kz

def illumination_amp(r, R, illum_key):
    # amplitude profile
    if illum_key == "uniform":
        return np.ones_like(r)
    if illum_key == "gauss_R":
        w = max(R, 1e-12)
        return np.exp(-(r/w)**2)
    if illum_key == "gauss_R2":
        w = max(R/2, 1e-12)
        return np.exp(-(r/w)**2)
    return np.ones_like(r)

def transmittance_r(r, f, D, element_key, phi_axis=0.0, axicon_dphi=8.0):
    """Complex amplitude transmittance t(r). Units: λ=1."""
    R = D/2.0
    P = (r <= R).astype(float)
    phi0 = float(phi_axis)
    k0 = 2*np.pi

    if element_key == "aperture":
        return (P*np.exp(1j*phi0)).astype(np.complex128)

    if element_key == "ring":
        rin = 0.7*R
        Pr = ((r <= R) & (r >= rin)).astype(float)
        return (Pr*np.exp(1j*phi0)).astype(np.complex128)

    if element_key == "axicon_phase":
        slope = axicon_dphi/max(R, 1e-12)
        phi = phi0 + slope*r
        return (P*np.exp(1j*phi)).astype(np.complex128)

    if element_key == "axicon_binary":
        slope = axicon_dphi/max(R, 1e-12)
        phi = phi0 + slope*r
        a = (np.cos(phi) > 0).astype(float)
        return (P*a).astype(np.complex128)

    if element_key == "lens_parabolic":
        # paraksjalna faza soczewki cienkiej: phi = -pi r^2 / f
        phi = phi0 + (-np.pi*(r**2)/max(f, 1e-12))
        return (P*np.exp(1j*phi)).astype(np.complex128)

    # spherical OPD to focus at z=f: OPD = sqrt(f^2+r^2) - f
    phi_sph = -k0*(np.sqrt(f*f + r*r) - f)

    if element_key == "lens_spherical":
        phi = phi0 + phi_sph
        return (P*np.exp(1j*phi)).astype(np.complex128)

    # zone plates based on spherical phase
    phi = phi0 + phi_sph
    if element_key == "zp_cont":
        a = 0.5*(1.0 + np.cos(phi))
        return (P*a).astype(np.complex128)
    if element_key == "zp_bin":
        a = (np.cos(phi) > 0).astype(float)
        return (P*a).astype(np.complex128)

    return (P*np.exp(1j*phi0)).astype(np.complex128)

def energy_selector(E2, H2, view_key):
    if view_key == "E":
        return E2
    if view_key == "H":
        return H2
    return 0.5*(E2 + H2)

def fwhm_center_or_ring1(x, y, eps=1e-15):
    """Always returns (fwhm, kind, clipped) unless profile is ~0 everywhere.
    kind: 'center' (on-axis peak), 'ring1' (first ring peak), 'global' (fallback)
    clipped: True if FWHM is truncated by x-range (no half-maximum crossing on one side)
    """
    x = np.asarray(x)
    y = np.asarray(y)
    y = np.maximum(y, 0.0)
    ymax = float(y.max())
    if ymax <= eps or x.size < 5:
        return 0.0, "global", True

    i0 = int(np.argmin(np.abs(x)))

    def is_local_max(i):
        return 0 < i < y.size-1 and (y[i] >= y[i-1]) and (y[i] >= y[i+1])

    if is_local_max(i0) and y[i0] > 0.1*ymax:
        ip = i0
        kind = "center"
    else:
        idx_pos = np.where(x >= 0)[0]
        xp = x[idx_pos]
        yp = y[idx_pos]
        d = np.diff(yp)
        cand = []
        for i in range(1, yp.size-1):
            if (d[i-1] >= 0) and (d[i] <= 0) and (yp[i] >= yp[i-1]) and (yp[i] >= yp[i+1]):
                cand.append(i)
        thr = 0.05*ymax
        cand = [i for i in cand if yp[i] >= thr]
        if cand:
            ip_pos = min(cand, key=lambda i: xp[i] if xp[i] > 0 else 1e99)
            ip = idx_pos[ip_pos]
            kind = "ring1"
        else:
            ip = int(np.argmax(y))
            kind = "global"

    ypeak = float(y[ip])
    half = 0.5*ypeak
    clipped = False

    # left crossing
    il = ip
    while il > 0 and y[il] >= half:
        il -= 1
    if il == 0 and y[il] >= half:
        x_left = float(x[0]); clipped = True
    else:
        x1, y1 = x[il], y[il]
        x2, y2 = x[il+1], y[il+1]
        t = (half - y1)/(y2 - y1 + eps)
        x_left = float(x1 + t*(x2-x1))

    # right crossing
    ir = ip
    while ir < y.size-1 and y[ir] >= half:
        ir += 1
    if ir == y.size-1 and y[ir] >= half:
        x_right = float(x[-1]); clipped = True
    else:
        x1, y1 = x[ir-1], y[ir-1]
        x2, y2 = x[ir], y[ir]
        t = (half - y1)/(y2 - y1 + eps)
        x_right = float(x1 + t*(x2-x1))

    fwhm = float(x_right - x_left)
    if fwhm < 0:
        fwhm = 0.0; clipped = True
    return fwhm, kind, clipped

In [4]:
# ----------------------------
# Hankel transforms (simple discrete quadrature)
# ----------------------------
def make_grids(r_max, Nr):
    r = np.linspace(0.0, r_max, Nr)
    dr = r[1]-r[0]
    k_max = np.pi/dr
    k = np.linspace(0.0, k_max, Nr)  # Nk = Nr
    dk = k[1]-k[0]
    return r, dr, k, dk

def hankel_forward(order, f_r, r, dr, k):
    # F(k) = 2π ∫ f(r) J_n(k r) r dr
    kr = np.outer(k, r)
    J = sp.j0(kr) if order == 0 else sp.j1(kr)
    return (2*np.pi) * (J @ (f_r * r)) * dr

def hankel_inverse(order, F_k, r, dk, k):
    # f(r) = 1/(2π) ∫ F(k) J_n(k r) k dk
    kr = np.outer(k, r)
    J = sp.j0(kr) if order == 0 else sp.j1(kr)
    return (1/(2*np.pi)) * (J.T @ (F_k * k)) * dk

In [5]:
# ----------------------------
# PL/EN i18n: labels + dropdown contents
# ----------------------------
I18N = {
  "pl": {
    "lang": "Język",
    "element": "Element",
    "mode": "Tryb obliczeń",
    "energy": "Co rysować",
    "illum": "Oświetlenie (włączane do transmitancji)",
    "f": "Ogniskowa f (λ)",
    "D": "Apertura D (średnica) (λ)",
    "phi": "Faza na osi φ_axis (rad)",
    "ax_dphi": "Aksikon: zakres fazy Δφ (rad) (0→R)",
    "Nr": "Nr (rozdzielczość transf. Hankela)",
    "ppL": "Próbkowanie r (pkt/λ)",
    "update": "Przelicz",
    "amp": "Transmitancja |t(x)|",
    "phase": "Faza arg(t) (rad)",
    "xz": "Mapa x–z: log10(u/umax)",
    "focus": "Przekrój w płaszczyźnie ogniskowej",
    "fwhm": "FWHM",
    "ring1": "pierścień 1",
    "global": "global",
    "clipped": "ucięte"
  },
  "en": {
    "lang": "Language",
    "element": "Element",
    "mode": "Computation mode",
    "energy": "What to plot",
    "illum": "Illumination (incl. in transmittance)",
    "f": "Focal length f (λ)",
    "D": "Aperture D (diameter) (λ)",
    "phi": "On-axis phase φ_axis (rad)",
    "ax_dphi": "Axicon: phase range Δφ (rad) (0→R)",
    "Nr": "Nr (Hankel transf. resolution)",
    "ppL": "Radial sampling(pts/λ)",
    "update": "Recompute",
    "amp": "Transmittance |t(x)|",
    "phase": "Phase arg(t) (rad)",
    "xz": "x–z map: log10(u/umax)",
    "focus": "Cross-section at focus plane",
    "fwhm": "FWHM",
    "ring1": "ring 1",
    "global": "global",
    "clipped": "clipped"
  }
}

ELEMENTS = {
  "lens_parabolic": {"pl": "1) Soczewka cienka paraboliczna (fazowa)", "en": "1) Parabolic thin lens (phase)"},
  "lens_spherical": {"pl": "2) Soczewka cienka sferyczna (fazowa)", "en": "2) Spherical thin lens (phase)"},
  "zp_cont": {"pl": "3) Ciągła płytka strefowa Fresnela (amplitudowa)", "en": "3) Fresnel zone plate (continuous, amplitude)"},
  "zp_bin": {"pl": "4) Binarna płytka strefowa Fresnela (amplitudowa)", "en": "4) Fresnel zone plate (binary, amplitude)"},
  "aperture": {"pl": "5) Bez soczewki (apertura kołowa)", "en": "5) No lens (circular aperture)"},
  "ring": {"pl": "6) Pierścień", "en": "6) Ring aperture"},
  "axicon_phase": {"pl": "7) Cienki aksikon (fazowy)", "en": "7) Thin axicon (phase)"},
  "axicon_binary": {"pl": "8) Aksikon z binarnymi strefami Fresnela", "en": "8) Axicon with binary Fresnel zones"}
}

MODES = {
  "scalar_fresnel": {"pl": "Skalarnie (Fresnel)", "en": "Scalar (Fresnel)"},
  "scalar_rs": {"pl": "Skalarnie (Rayleigh–Sommerfeld)", "en": "Scalar (Rayleigh–Sommerfeld)"},
  "radial": {"pl": "Polaryzacja radialna (RS)", "en": "Radial polarization (RS)"},
  "azimuthal": {"pl": "Polaryzacja azymutalna (RS)", "en": "Azimuthal polarization (RS)"},
  "lin_x": {"pl": "Polaryzacja liniowa x (RS)", "en": "Linear x polarization (RS)"},
  "lin_y": {"pl": "Polaryzacja liniowa y (RS)", "en": "Linear y polarization (RS)"}
}

ENERGY_VIEWS = {
  "total": {"pl": "Całkowita gęstość energii u(E)+u(H)", "en": "Total energy density u(E)+u(H)"},
  "E": {"pl": "Tylko elektryczna u(E)", "en": "Electric only u(E)"},
  "H": {"pl": "Tylko magnetyczna u(H)", "en": "Magnetic only u(H)"}
}

ILLUMS = {
  "uniform": {"pl": "Jednorodne", "en": "Uniform"},
  "gauss_R": {"pl": "Gauss: w = R", "en": "Gaussian: w = R"},
  "gauss_R2": {"pl": "Gauss: w = R/2", "en": "Gaussian: w = R/2"}
}

def options_from_dict(d, lang):
    # returns list of (label, key) for widgets
    return [(v[lang], k) for k, v in d.items()]


In [6]:
# ----------------------------
# Propagation for 6 modes
# ----------------------------
def propagate_u(
    field0_r, r, dr, k, dk,
    f, z_min, Nz,
    mode_key="scalar_rs",
    energy_view_key="total",
    eta0=1.0,
    k0=2*np.pi,
):
    z_end = max(float(f), z_min + 1e-6)
    z = np.linspace(z_min, z_end, Nz)

    Nr = r.size
    logu = np.empty((Nz, Nr), dtype=float)

    # initial spectra
    if mode_key in ("scalar_fresnel", "scalar_rs", "lin_x", "lin_y"):
        U_k0 = hankel_forward(0, field0_r, r, dr, k)
    elif mode_key == "radial":
        Er_k0 = hankel_forward(1, field0_r, r, dr, k)
    elif mode_key == "azimuthal":
        Ephi_k0 = hankel_forward(1, field0_r, r, dr, k)
    else:
        U_k0 = hankel_forward(0, field0_r, r, dr, k)

    umax = 0.0
    u_last = None

    for i, zi in enumerate(z):
        if mode_key == "scalar_fresnel":
            H = H_fresnel(k0, zi, k)
            U_r = hankel_inverse(0, U_k0*H, r, dk, k)
            E2 = np.abs(U_r)**2
            H2 = np.abs(U_r)**2
            u = energy_selector(E2, H2, "total" if energy_view_key=="total" else ("E" if energy_view_key=="E" else "H"))

        elif mode_key == "scalar_rs":
            H, _kz = H_rs(k0, zi, k)
            U_r = hankel_inverse(0, U_k0*H, r, dk, k)
            E2 = np.abs(U_r)**2
            H2 = np.abs(U_r)**2
            u = energy_selector(E2, H2, "total" if energy_view_key=="total" else ("E" if energy_view_key=="E" else "H"))

        elif mode_key == "radial":
            H, kz = H_rs(k0, zi, k)
            Er_k = Er_k0 * (kz/k0) * H
            Ez_k = -Er_k0 * (k/k0) * H
            Hphi_k = ((k/k0)*Ez_k - (kz/k0)*Er_k) / eta0

            Er_r = hankel_inverse(1, Er_k, r, dk, k)
            Ez_r = hankel_inverse(0, Ez_k, r, dk, k)
            Hphi_r = hankel_inverse(1, Hphi_k, r, dk, k)

            E2 = np.abs(Er_r)**2 + np.abs(Ez_r)**2
            H2 = np.abs(Hphi_r)**2
            u = energy_selector(E2, H2, "total" if energy_view_key=="total" else ("E" if energy_view_key=="E" else "H"))

        elif mode_key == "azimuthal":
            H, kz = H_rs(k0, zi, k)
            Ephi_k = Ephi_k0 * H
            Hr_k = (-(kz/k0) * Ephi_k) / eta0
            Hz_k = ((k/k0) * Ephi_k) / eta0

            Ephi_r = hankel_inverse(1, Ephi_k, r, dk, k)
            Hr_r = hankel_inverse(1, Hr_k, r, dk, k)
            Hz_r = hankel_inverse(0, Hz_k, r, dk, k)

            E2 = np.abs(Ephi_r)**2
            H2 = np.abs(Hr_r)**2 + np.abs(Hz_r)**2
            u = energy_selector(E2, H2, "total" if energy_view_key=="total" else ("E" if energy_view_key=="E" else "H"))

        elif mode_key in ("lin_x", "lin_y"):
            # axisymmetric surrogate: propagate scalar RS envelope and treat it as transverse
            H, _kz = H_rs(k0, zi, k)
            U_r = hankel_inverse(0, U_k0*H, r, dk, k)
            E2 = np.abs(U_r)**2
            H2 = np.abs(U_r)**2
            u = energy_selector(E2, H2, "total" if energy_view_key=="total" else ("E" if energy_view_key=="E" else "H"))

        else:
            u = np.zeros_like(r)

        umax = max(umax, float(u.max()))
        logu[i, :] = np.log10(u + 1e-15)
        if i == Nz-1:
            u_last = u.copy()

    umax = umax + 1e-15
    logu = np.log10((10.0**logu)/umax + 1e-15)
    u_focus = u_last / (float(u_last.max()) + 1e-15)
    return z, logu, u_focus

In [7]:
# ----------------------------
# UI + plotting
# ----------------------------
lang = widgets.ToggleButtons(options=[("PL","pl"),("EN","en")], value="pl")

element_w = widgets.Dropdown(options=options_from_dict(ELEMENTS, lang.value), value="lens_parabolic")
mode_w    = widgets.Dropdown(options=options_from_dict(MODES, lang.value), value="scalar_rs")
energy_w  = widgets.Dropdown(options=options_from_dict(ENERGY_VIEWS, lang.value), value="total")
illum_w   = widgets.Dropdown(options=options_from_dict(ILLUMS, lang.value), value="uniform")
cmap_w = widgets.Dropdown(
    options=["viridis","magma","inferno","plasma","cividis","turbo","gray","gnuplot2"],
    value="viridis",
    description="cmap:"
)



f_w = widgets.FloatSlider(value=200.0, min=10.0, max=1500.0, step=1.0, readout_format=".1f")
D_w = widgets.FloatSlider(value=400.0, min=10.0, max=2400.0, step=1.0, readout_format=".1f")
phi_w = widgets.FloatSlider(value=0.0, min=0.0, max=8.0, step=0.001, readout_format=".3f")
ax_dphi_w = widgets.FloatSlider(value=8.0, min=0.0, max=40.0, step=0.001, readout_format=".3f")

Nr_w = widgets.IntSlider(value=512, min=256, max=4096*4, step=256)
ppL_w = widgets.IntSlider(value=1, min=1, max=50, step=1)

btn = widgets.Button(description="", button_style="primary")
out = widgets.Output()

def relabel():
    t = I18N[lang.value]
    btn.description = t["update"]
    element_w.description = t["element"] + ":"
    mode_w.description = t["mode"] + ":"
    energy_w.description = t["energy"] + ":"
    illum_w.description = t["illum"] + ":"
    f_w.description = t["f"] + ":"
    D_w.description = t["D"] + ":"
    phi_w.description = t["phi"] + ":"
    ax_dphi_w.description = t["ax_dphi"] + ":"
    Nr_w.description = t["Nr"] + ":"
    ppL_w.description = t["ppL"] + ":"

def refresh_dropdown_labels():
    # Keep selected keys, only change displayed labels
    el_key = element_w.value
    mo_key = mode_w.value
    en_key = energy_w.value
    il_key = illum_w.value
    element_w.options = options_from_dict(ELEMENTS, lang.value)
    mode_w.options    = options_from_dict(MODES, lang.value)
    energy_w.options  = options_from_dict(ENERGY_VIEWS, lang.value)
    illum_w.options   = options_from_dict(ILLUMS, lang.value)
    element_w.value = el_key
    mode_w.value = mo_key
    energy_w.value = en_key
    illum_w.value = il_key

def compute_and_plot(_=None):
    with out:
        clear_output(wait=True)

        t = I18N[lang.value]
        k0 = 2*np.pi

        f = float(f_w.value)
        D = float(D_w.value)
        R = D/2.0


        

        # r_max: safety margin
        
        Nr = int(Nr_w.value)
        ppL = int(ppL_w.value)
        r_max = min(2.0*R, Nr/ppL)
        

        # Nz limits based on Nr
        Nz = 220
        if Nr > 4096:
            Nz = min(Nz, 5)
        elif Nr > 1024:
            Nz = min(Nz, 25)

        r, dr, k, dk = make_grids(r_max, Nr)

        t_r = transmittance_r(r, f, D, element_w.value, phi_axis=float(phi_w.value), axicon_dphi=float(ax_dphi_w.value))
        A = illumination_amp(r, R, illum_w.value)
        field0 = (A * t_r).astype(np.complex128)

        z, logu_rz, u_focus_r = propagate_u(
            field0, r, dr, k, dk,
            f=f, z_min=0.5, Nz=Nz,
            mode_key=mode_w.value,
            energy_view_key=energy_w.value,
            eta0=1.0, k0=k0
        )

        # mirror to x
        x = build_mirrored_x_from_r(r)
        tx = mirror_profile_to_x(field0)
        amp_x = np.abs(tx)
        phase_x = np.angle(tx)
        phase_x[amp_x <= 1e-15] = np.nan

        logu_xz = np.concatenate((logu_rz[:, :0:-1], logu_rz), axis=1)
        u_focus_x = mirror_profile_to_x(u_focus_r)

        # x sampling for FWHM only
        
        if ppL > 1:
            dx = 1.0/ppL
            x_plot = np.arange(x[0], x[-1] + 0.5*dx, dx)
            u_focus_x_plot = np.interp(x_plot, x, u_focus_x)
        else:
            x_plot = x
            u_focus_x_plot = u_focus_x

        fwhm, kind, clipped = fwhm_center_or_ring1(x_plot, u_focus_x_plot)
        if kind == "center":
            kind_txt = t["fwhm"]
        elif kind == "ring1":
            kind_txt = f"{t['fwhm']:.2} ({t['ring1']})"
        else:
            kind_txt = f"{t['fwhm']:.2} ({t['global']})"
        clip_txt = f" ({t['clipped']})" if clipped else ""
        fwhm_txt = f"{kind_txt}={fwhm:.2f} λ{clip_txt}"

        # plot
        fig = plt.figure(figsize=(11, 12))
        gs = fig.add_gridspec(4, 1, height_ratios=[1.0, 1.0, 2.2, 1.2], hspace=0.38)

        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
        ax3 = fig.add_subplot(gs[2, 0])
        ax4 = fig.add_subplot(gs[3, 0], sharex=ax1)

        ax1.plot(x, amp_x, lw=1.6)
        ax1.set_ylabel("|t(x)|")
        ax1.set_title(t["amp"])
        ax1.grid(True, alpha=0.25)

        ax2.plot(x, phase_x, lw=1.4)
        ax2.set_ylabel("arg(t) (rad)")
        ax2.set_xlabel("x (λ)")
        ax2.set_ylim(-np.pi, np.pi)
        ax2.grid(True, alpha=0.25)

        extent = [x[0], x[-1], z[-1], z[0]]
        im = ax3.imshow(logu_xz, aspect="auto", extent=extent, interpolation="nearest",cmap=cmap_w.value)
        ax3.set_title(f"{t['xz']} | {MODES[mode_w.value][lang.value]} | {ENERGY_VIEWS[energy_w.value][lang.value]}")
        ax3.set_xlabel("x (λ)")
        ax3.set_ylabel("z (λ)")
        ax3.axhline(f, ls="--", lw=1.2)
        cbar = fig.colorbar(im, ax=ax3, orientation="horizontal", pad=0.22)
        cbar.set_label("log10(u/umax)")

        ax4.plot(x_plot, u_focus_x_plot, lw=1.7)
        ax4.set_title(f"{t['focus']} — {fwhm_txt}")
        ax4.set_xlabel("x [λ]")
        ax4.set_ylabel("u/umax")
        ax4.grid(True, alpha=0.25)

        plt.show()

def on_lang_change(change):
    relabel()
    refresh_dropdown_labels()

lang.observe(on_lang_change, names="value")
btn.on_click(compute_and_plot)

relabel()

controls = widgets.VBox([
    widgets.HBox([widgets.Label("Language / Język:"), lang]),
    element_w,
    mode_w,
    widgets.HBox([energy_w, illum_w]),
    f_w, D_w, phi_w, ax_dphi_w,cmap_w,
    widgets.HBox([Nr_w, ppL_w]),
    btn
])
controls.layout = widgets.Layout(width='100%')

display(controls, out)
compute_and_plot()

Output()